# 第 25 章：不确定度、解释性与科学可信度

这个 notebook 用一个教学版光度红移数据集演示：

- bootstrap 区间能告诉我们什么，又不能告诉我们什么
- permutation importance 如何做最小特征重要性分析
- 局部解释更像模型行为说明，而不是物理因果证明
- 为什么样本一旦跑出训练分布，窄区间也可能不可靠


In [ ]:
from __future__ import annotations

import csv
import math
import random
from pathlib import Path

DATA_PATH = Path("../../data/small/photoz_trust_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            "galaxy_id": raw["galaxy_id"],
            "u_g": float(raw["u_g"]),
            "g_r": float(raw["g_r"]),
            "r_i": float(raw["r_i"]),
            "i_z": float(raw["i_z"]),
            "specz": float(raw["specz"]),
            "role": raw["role"],
        })

print(f"Loaded {len(rows)} photo-z samples from {DATA_PATH.name}")
role_counts = {}
for row in rows:
    role_counts[row["role"]] = role_counts.get(row["role"], 0) + 1
print("role counts:", role_counts)
print("first row:", rows[0])


In [ ]:
feature_names = ["u_g", "g_r", "r_i", "i_z"]
train_rows = [row for row in rows if row["role"] == "train"]
test_rows = [row for row in rows if row["role"] == "test"]
edge_rows = [row for row in rows if row["role"] == "edge"]

print("train/test/edge sizes:", len(train_rows), len(test_rows), len(edge_rows))
print("edge ids:", [row["galaxy_id"] for row in edge_rows])


In [ ]:
def solve_linear_system(matrix, vector):
    size = len(vector)
    augmented = [row[:] + [value] for row, value in zip(matrix, vector)]

    for col in range(size):
        pivot_row = max(range(col, size), key=lambda row_index: abs(augmented[row_index][col]))
        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot_value = augmented[col][col]
        if abs(pivot_value) < 1e-12:
            raise ValueError("Singular matrix")
        for j in range(col, size + 1):
            augmented[col][j] /= pivot_value
        for row in range(size):
            if row == col:
                continue
            factor = augmented[row][col]
            for j in range(col, size + 1):
                augmented[row][j] -= factor * augmented[col][j]

    return [augmented[row][size] for row in range(size)]


def design_vector(row):
    return [1.0] + [row[name] for name in feature_names]


def fit_ridge_linear_regression(data_rows, ridge=1e-6):
    design = [design_vector(row) for row in data_rows]
    targets = [row["specz"] for row in data_rows]
    parameter_count = len(design[0])
    matrix = []
    vector = []

    for i in range(parameter_count):
        current_row = []
        for j in range(parameter_count):
            value = sum(design_row[i] * design_row[j] for design_row in design)
            if i == j and i != 0:
                value += ridge
            current_row.append(value)
        matrix.append(current_row)
        vector.append(sum(design_row[i] * target for design_row, target in zip(design, targets)))

    return solve_linear_system(matrix, vector)


def predict(coefficients, row):
    return sum(coefficient * value for coefficient, value in zip(coefficients, design_vector(row)))


def rmse(data_rows, coefficients):
    squared_errors = [(predict(coefficients, row) - row["specz"]) ** 2 for row in data_rows]
    return math.sqrt(sum(squared_errors) / len(squared_errors))


coefficients = fit_ridge_linear_regression(train_rows)
print("coefficients:", [round(value, 4) for value in coefficients])
print("train RMSE:", round(rmse(train_rows, coefficients), 4))
print("test RMSE:", round(rmse(test_rows, coefficients), 4))
print("edge RMSE:", round(rmse(edge_rows, coefficients), 4))


In [ ]:
evaluation_rows = test_rows + edge_rows
print("Predictions on test and edge samples:")
for row in evaluation_rows:
    prediction = predict(coefficients, row)
    error = prediction - row["specz"]
    print(
        row["galaxy_id"],
        row["role"],
        f"true={row["specz"]:.3f}",
        f"pred={prediction:.3f}",
        f"error={error:.3f}",
    )


In [ ]:
def percentile(sorted_values, fraction):
    index = int(fraction * len(sorted_values))
    index = max(0, min(index, len(sorted_values) - 1))
    return sorted_values[index]


random.seed(0)
bootstrap_predictions = {row["galaxy_id"]: [] for row in evaluation_rows}
for _ in range(200):
    sampled_train = [random.choice(train_rows) for _ in train_rows]
    sampled_coefficients = fit_ridge_linear_regression(sampled_train)
    for row in evaluation_rows:
        bootstrap_predictions[row["galaxy_id"]].append(predict(sampled_coefficients, row))

print("Bootstrap intervals:")
for row in evaluation_rows:
    values = sorted(bootstrap_predictions[row["galaxy_id"]])
    lower = percentile(values, 0.16)
    upper = percentile(values, 0.84)
    covered = lower <= row["specz"] <= upper
    print(
        row["galaxy_id"],
        row["role"],
        f"interval=({lower:.3f}, {upper:.3f})",
        f"width={upper - lower:.3f}",
        f"contains_truth={covered}",
    )


In [ ]:
baseline_test_rmse = rmse(test_rows, coefficients)
print("Permutation importance on test set:")
for feature_name in feature_names:
    shuffled_values = [row[feature_name] for row in test_rows]
    shuffled_values = shuffled_values[1:] + shuffled_values[:1]
    modified_rows = []
    for row, value in zip(test_rows, shuffled_values):
        modified = dict(row)
        modified[feature_name] = value
        modified_rows.append(modified)
    shuffled_rmse = rmse(modified_rows, coefficients)
    print(feature_name, round(shuffled_rmse - baseline_test_rmse, 4))


In [ ]:
training_feature_means = {name: sum(row[name] for row in train_rows) / len(train_rows) for name in feature_names}
target_row = next(row for row in evaluation_rows if row["galaxy_id"] == "E001")
base_prediction = predict(coefficients, target_row)
print("Local explanation intuition for E001:")
print("  base prediction:", round(base_prediction, 3))
for feature_name in feature_names:
    modified = dict(target_row)
    modified[feature_name] = training_feature_means[feature_name]
    modified_prediction = predict(coefficients, modified)
    delta = base_prediction - modified_prediction
    print(f"  {feature_name}: delta={delta:.3f}")
print("  Note: these deltas are a SHAP-like intuition, not exact Shapley values.")


In [ ]:
feature_means = {}
feature_stds = {}
for feature_name in feature_names:
    values = [row[feature_name] for row in train_rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    feature_means[feature_name] = mean_value
    feature_stds[feature_name] = math.sqrt(variance) or 1.0


def standardized_vector(row):
    return [
        (row[feature_name] - feature_means[feature_name]) / feature_stds[feature_name]
        for feature_name in feature_names
    ]


train_vectors = [standardized_vector(row) for row in train_rows]
print("Distance to training support:")
for row in evaluation_rows:
    vector = standardized_vector(row)
    nearest_distance = min(
        math.sqrt(sum((left - right) ** 2 for left, right in zip(vector, train_vector)))
        for train_vector in train_vectors
    )
    print(row["galaxy_id"], row["role"], round(nearest_distance, 3))
print("Observation: edge samples are far outside the training support even when bootstrap intervals are not extremely wide.")


In [ ]:
try:
    from sklearn.inspection import permutation_importance
    from sklearn.linear_model import Ridge
except ModuleNotFoundError:
    print("scikit-learn 未安装；已跳过标准库可信度示例。")
else:
    train_matrix = [[row[name] for name in feature_names] for row in train_rows]
    test_matrix = [[row[name] for name in feature_names] for row in test_rows]
    train_targets = [row["specz"] for row in train_rows]
    model = Ridge(alpha=1e-6)
    model.fit(train_matrix, train_targets)
    importance = permutation_importance(model, test_matrix, [row["specz"] for row in test_rows], n_repeats=20, random_state=0)
    print("sklearn permutation importances:")
    for feature_name, value in sorted(zip(feature_names, importance.importances_mean), key=lambda item: item[1], reverse=True):
        print(feature_name, round(value, 4))
